## Importações

In [0]:
# Importando as bibliotecas
import pyspark.sql.functions as F
from delta.tables import DeltaTable

In [0]:
df_bronze = spark.read.table('training.bronze.precos_acoes_b3')

## Tratamentos

In [0]:
# Pegando dados dos ultimos 3 dias
novos_dados = df_bronze.filter(F.col('data_ingestao')  >= F.current_date() - 3)

In [0]:
# Ajuste nos tipos das datas
novos_dados = df_bronze.withColumn('data_referencia', F.to_date(F.col('data_referencia'), 'yyyy-MM-dd'))
novos_dados = novos_dados.withColumn('data_ingestao', F.to_date(F.col('data_ingestao'), 'yyyy-MM-dd'))

In [0]:
# Ajuste nos nomes das colunas (Qaundo aplicavel
novos_dados = novos_dados.withColumnRenamed('data_ingestao', 'data_ingestao_db')

In [0]:
# Definindp a data de ingestao na Silver
novos_dados = novos_dados.withColumn('data_extracao', F.current_date())

## Escrita de dados

In [0]:
# # Colocando os dados na Silver (Unica execução)
# novos_dados.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable('training.silver.precos_acoes_b3')

In [0]:
from delta.tables import DeltaTable

# Referência da tabela destino na camada Silver
tabela_silver = DeltaTable.forName(spark, "training.silver.precos_acoes_b3")

# Executando o Merge mantendo a data_ingestao original no update
(
    tabela_silver.alias("silver")
    .merge(
            source=novos_dados.alias("novos_dados"),
            condition="novos_dados.id = silver.id AND novos_dados.ticker = silver.ticker AND novos_dados.data_referencia = silver.data_referencia"
        )
    .whenMatchedUpdate(
        set={
            "data_ingestao": "silver.data_ingestao"
        }
    )
    .whenNotMatchedInsertAll()  # Insere a linha inteira (incluindo a data_ingestao da novos_dados) se não existir
    .execute()
)